# 02 — Text Cleaning and Preprocessing

## Objective
Transform raw YouTube comments into clean, language-filtered, normalised text ready for NLP models.

## Cleaning decisions and rationale

| Decision | Rationale |
|----------|----------|
| **Decode HTML entities** | YouTube comments contain ``&amp;``, ``&lt;`` — these must be resolved to actual characters |
| **Remove URLs** | Carry no lexical sentiment signal; fragment topic clusters |
| **Remove @-mentions** | @-mentions add noise and are not meaningful for sentiment |
| **Extract then remove emojis** | Emojis are strong sentiment signals but tokenizers handle them poorly; stored as a separate feature |
| **Strip bullet/list markers** | YouTube comments sometimes start with ``- `` or ``* `` from copy-paste |
| **Collapse repeating chars** | "nooooo" → "nooo" to reduce vocabulary sparsity |
| **Language filter** | Only Spanish and English kept; the BERT models are language-specific |
| **Deduplicate by text hash** | SHA-256 hash catches cross-post duplicates |

## Business value
Garbage in → garbage out. Rigorous cleaning directly improves model accuracy. The emoji-as-feature design lets us later quantify whether emoji usage correlates with sentiment.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import preprocess_dataframe, clean_text, detect_language, extract_emojis
from src.data_collection import load_collected
from src.utils import save_dataframe

### Load raw data

In [ ]:
df = load_collected(format='parquet')
print(f"Raw comments loaded: {len(df)}")
if not df.empty:
    display(df[['text', 'teams', 'video_title']].head(3))

### Preview cleaning on a sample

In [ ]:
if not df.empty:
    sample = df['text'].iloc[0]
    print("=== RAW ===")
    print(sample)
    print("\n=== CLEANED ===")
    print(clean_text(sample))
    print("\n=== LANGUAGE ===")
    lang, conf = detect_language(sample)
    print(f"{lang} (confidence: {conf:.2f})")
    print("\n=== EMOJIS FOUND ===")
    emojis = extract_emojis(sample)
    print(emojis if emojis else "(none)")

### Run full preprocessing pipeline

In [ ]:
if not df.empty:
    df_clean = preprocess_dataframe(df)
    print(f"After preprocessing: {len(df_clean)} comments")
    print(f"Dropped: {len(df) - len(df_clean)} comments")
    display(df_clean[['text_clean', 'language', 'n_emojis', 'teams']].head())

### Language distribution

In [ ]:
if not df_clean.empty:
    lang_dist = df_clean['language'].value_counts()
    display(lang_dist)
    import matplotlib.pyplot as plt
    lang_dist.plot(kind='bar', title='Language Distribution')
    plt.xticks(rotation=0)
    plt.show()

### Save preprocessed data

In [ ]:
if not df_clean.empty:
    save_dataframe(df_clean, str(project_root / 'data' / 'processed' / 'preprocessed'), format='parquet')
    print("Saved to data/processed/preprocessed.parquet")

---
**Next step**: [03 — Sentiment Analysis](03_analisis_sentimiento.ipynb)